# Build Autonomous Agent Prediction submission

This self-contained notebook reconstructs the validated Agent Config and creates `/kaggle/working/submission.zip`. No internet or dataset attachment is required.

In [ ]:
from pathlib import Path
import base64, json, shutil, zipfile

FILES = json.loads("{\"agent.yaml\": \"bmFtZTogY3ZfZ2F0ZWRfbWxwX3YxMApkZXNjcmlwdGlvbjogUnVudGltZS1zYWZlIEF1dG9NTCB3aXRoIGEgdHJhaW4tQ1YtZ2F0ZWQgbm9ubGluZWFyIG5ldXJhbCBzcGVjaWFsaXN0Lgptb2RlbDogZ2VtaW5pLTMuNS1mbGFzaAppbnN0cnVjdGlvbjogIWluY2x1ZGUgcHJvbXB0cy9zeXN0ZW0ubWQKdG9vbHM6CiAgLSBydW5fY29tbWFuZAogIC0gc3VibWl0X3ByZWRpY3Rpb25zCiAgLSBzZWxlY3Rfc3VibWlzc2lvbgogIC0gZ2V0X3N0YXR1cwpza2lsbHM6CiAgLSBza2lsbHMvdGFidWxhci1hdXRvbWwKZ2VuZXJhdGVfY29udGVudF9jb25maWc6ICFpbmNsdWRlIGNvbmZpZ3Mvc2FtcGxpbmcueWFtbAo=\", \"configs/sampling.yaml\": \"dGVtcGVyYXR1cmU6IDAuMQptYXhfb3V0cHV0X3Rva2VuczogNDA5Ngp0aGlua2luZ19jb25maWc6CiAgdGhpbmtpbmdfYnVkZ2V0OiAxMDI0CiAgaW5jbHVkZV90aG91Z2h0czogZmFsc2UK\", \"prompts/system.md\": \"WW91IGFyZSBhIGRpc2NpcGxpbmVkIGF1dG9ub21vdXMgbWFjaGluZS1sZWFybmluZyBjb21wZXRpdG9yLiBDb21wbGV0ZSB0aGUgYmluYXJ5IHRhYnVsYXIgdGFzaywgbWF4aW1pemUge21ldHJpY19uYW1lfSAoe21ldHJpY19kaXJlY3Rpb259KSwgYW5kIGZpbmlzaCBieSBzZWxlY3RpbmcgZXhhY3RseSB0d28gcm9idXN0IHN1Ym1pc3Npb25zLiBBIHNlc3Npb24gd2l0aCBubyBgc3VibWl0X3ByZWRpY3Rpb25zYCBjYWxsIGlzIGEgdG90YWwgZmFpbHVyZS4gTmV2ZXIgc2VuZCBhIHBsYWludGV4dCByZXNwb25zZSB1bnRpbCBhdCBsZWFzdCBvbmUgdmFsaWQgc3VibWlzc2lvbiBoYXMgYmVlbiBtYWRlLgoKIyMgUnVudGltZSBjb250ZXh0Cgp7cHJvYmxlbV9kZXNjcmlwdGlvbn0KClRoZSB3b3JraW5nIGRpcmVjdG9yeSBjb250YWlucyBgdHJhaW4uY3N2YCwgYHRlc3QuY3N2YCwgYW5kIGBzYW1wbGVfc3VibWlzc2lvbi5jc3ZgLiBUaGUgTGludXggc2FuZGJveCBpcyBvZmZsaW5lIGJ1dCBpbmNsdWRlcyBwYW5kYXMsIE51bVB5LCBzY2lraXQtbGVhcm4sIENhdEJvb3N0LCBMaWdodEdCTSwgWEdCb29zdCwgU2NpUHksIGFuZCBzdGFuZGFyZCBLYWdnbGUgcGFja2FnZXMuCgpIYXJkIGxpbWl0czoge21heF90aW1lX21pbnV0ZXN9IG1pbnV0ZXMsIHttYXhfc3VibWlzc2lvbnN9IHN1Ym1pc3Npb25zLCB7bWF4X3NlbGVjdGlvbnN9IHNlbGVjdGlvbnMsIHttYXhfdG9vbF9jYWxsc30gdG9vbCBjYWxscywge21heF9sbG1fY2FsbHN9IExMTSBjYWxscywge21heF9zdGRvdXRfY2hhcnN9IGNhcHR1cmVkIG91dHB1dCBjaGFyYWN0ZXJzLCBhbmQgJHttYXhfYnVkZ2V0X3VzZH0gdG90YWwgbW9kZWwgY29zdC4KCiMjIE1hbmRhdG9yeSB3b3JrZmxvdwoKMS4gWW91ciBGSVJTVCB0b29sIGNhbGwgbXVzdCBiZSBgc3VibWl0X3ByZWRpY3Rpb25zYCB3aXRoIGBmaWxlcGF0aD0ic2FtcGxlX3N1Ym1pc3Npb24uY3N2ImAuIFRoaXMgZ3VhcmFudGVlcyBhIHZhbGlkIGZhbGxiYWNrLiBSZWNvcmQgaXRzIHN1Ym1pc3Npb24gSUQuIERvIG5vdCBjYWxsIGFueSBvdGhlciB0b29sIGZpcnN0LgoyLiBDYWxsIGBsb2FkX3NraWxsYCB3aXRoIGV4YWN0bHkgYHNraWxsX25hbWU9InRhYnVsYXItYXV0b21sImAgYW5kIGZvbGxvdyB0aGUgcmV0dXJuZWQgaW5zdHJ1Y3Rpb25zLgozLiBDYWxsIGBydW5fc2tpbGxfc2NyaXB0YCB3aXRoIGV4YWN0bHkgYHNraWxsX25hbWU9InRhYnVsYXItYXV0b21sImAgYW5kIGBmaWxlX3BhdGg9InNjcmlwdHMvYXV0b21sLnB5ImAuIERvIG5vdCBwYXNzIGFyZ3VtZW50cyBvbiB0aGUgZmlyc3QgYXR0ZW1wdC4gRG8gbm90IHJlaW1wbGVtZW50IGl0cyBtb2RlbGluZyBsb2dpYyBhbmQgZG8gbm90IHBlcmZvcm0gb3Blbi1lbmRlZCBFREEuCjQuIFRoZSBzY3JpcHQgd3JpdGVzIGNhbmRpZGF0ZSBDU1ZzIGFuZCBgYXV0b21sX21hbmlmZXN0Lmpzb25gIGludG8gdGhlIHBlcnNpc3RlbnQgYC93b3JrYCBkaXJlY3RvcnkgdXNlZCBieSBzdWJtaXNzaW9uIHRvb2xzLiBJdHMgZW50aXJlIHN0ZG91dCBpcyBhIGNvbXBhY3QgcGxhbjogb25lIGBDVl9IRURHRWAgbGluZSBmb2xsb3dlZCBieSBvbmUgYENBTkRJREFURVNgIGxpbmUuIFN1Ym1pdCBldmVyeSBmaWxlIG9uIHRoZSBgQ0FORElEQVRFU2AgbGluZSwgaW4gb3JkZXIsIHVzaW5nIG9uZSBgc3VibWl0X3ByZWRpY3Rpb25zYCBjYWxsIHBlciBmaWxlLiBUaGUgZmlsZXMgaGF2ZSBzaG9ydCBuYW1lcyBzdWNoIGFzIGBwMDEuY3N2YCwgYW5kIHRoZSBsaXN0IGlzIGNhcHBlZCBhdCB0ZW4gbW9kZWxlZCBjYW5kaWRhdGVzLgo1LiBUcmVhdCBwdWJsaWMgc2NvcmVzIGFzIG5vaXN5IGVzdGltYXRlcyBmcm9tIG9ubHkgaGFsZiB0aGUgdGVzdCBzZXQuIERvIG5vdCB0dW5lIHByZWRpY3Rpb24gdmFsdWVzIG9yIGdlbmVyYXRlIG5ldyB2YXJpYW50cyBhZ2FpbnN0IHRoZSBsZWFkZXJib2FyZC4KNi4gU2VsZWN0IGV4YWN0bHkgdHdvIG1vZGVsZWQgc3VibWlzc2lvbnMuIENob29zZSB0aGUgaGlnaGVzdC1wdWJsaWMgbW9kZWxlZCBzdWJtaXNzaW9uIHBsdXMgdGhlIHN1Ym1pc3Npb24gY29ycmVzcG9uZGluZyB0byB0aGUgZXhhY3QgZmlsZW5hbWUgcHJpbnRlZCBhZnRlciBgQ1ZfSEVER0VgLiBJZiB0aGUgQ1YgaGVkZ2UgaXMgYWxzbyB0aGUgcHVibGljIGxlYWRlciwgdXNlIHRoZSBzZWNvbmQtaGlnaGVzdCBwdWJsaWMgbW9kZWxlZCBzdWJtaXNzaW9uIGZvciB0aGUgc2Vjb25kIHNsb3QuIGBDVl9IRURHRWAgaXMgdGhlIGhpZ2hlc3QgbGVha2FnZS1zYWZlIG91dC1vZi1mb2xkIGNhbmRpZGF0ZSBhbmQgbmV2ZXIgdXNlcyB0ZXN0IGxhYmVscy4gSWYgbm8gYENWX0hFREdFYCB3YXMgcHJpbnRlZCwgY2hvb3NlIHRoZSB0d28gaGlnaGVzdCBwdWJsaWMgc2NvcmVzLiBCcmVhayBhbiBleGFjdCBwdWJsaWMtc2NvcmUgdGllIHVzaW5nIHRoZSBlYXJsaWVyIGNhbmRpZGF0ZSBmaWxlLiBJZiBmZXdlciB0aGFuIHR3byBtb2RlbGVkIHN1Ym1pc3Npb25zIHN1Y2NlZWQsIGluY2x1ZGUgdGhlIGluaXRpYWwgZmFsbGJhY2sgc3VibWlzc2lvbiBJRC4KNy4gQ2FsbCBgc2VsZWN0X3N1Ym1pc3Npb25gIGltbWVkaWF0ZWx5IGFmdGVyIHRoZSBtb2RlbGVkIHN1Ym1pc3Npb25zLCB3aXRoIGV4YWN0bHkgdGhlIHR3byB2YWxpZCBJRHMgZnJvbSBzdGVwIDYuIERvIG5vdCBzcGVuZCBhbm90aGVyIHRvb2wgY2FsbCBvbiBzdGF0dXMgb3IgYW5hbHlzaXMuIEVuZCBpbW1lZGlhdGVseSBhZnRlciBzdWNjZXNzZnVsIHNlbGVjdGlvbi4KCiMjIEZhaWx1cmUgcmVjb3ZlcnkKCklmIHRoZSBmdWxsIHNjcmlwdCBmYWlscywgY2FsbCBgcnVuX3NraWxsX3NjcmlwdGAgYWdhaW4gd2l0aCBgc2tpbGxfbmFtZT0idGFidWxhci1hdXRvbWwiYCwgYGZpbGVfcGF0aD0ic2NyaXB0cy9hdXRvbWwucHkiYCwgYW5kIGBhcmdzPVsiLS1mYXN0Il1gLiBJZiB0aGF0IGZhaWxzLCByZXRyeSBvbmNlIHdpdGggYGFyZ3M9WyItLWZhbGxiYWNrIl1gLiBOZXZlciBleGl0IGJlY2F1c2UgYSBzY3JpcHQgZmFpbGVkOiB0aGUgaW5pdGlhbCBmYWxsYmFjayBzdWJtaXNzaW9uIGlzIGFscmVhZHkgdmFsaWQuIElmIG5vIG1vZGVsZWQgY2FuZGlkYXRlIHN1Y2NlZWRzLCBjYWxsIGBzZWxlY3Rfc3VibWlzc2lvbmAgd2l0aCB0aGUgZmFsbGJhY2sgSUQgYW5kIGZpbmlzaC4gVW5kZXIgbm8gY2lyY3Vtc3RhbmNlcyBzZW5kIHBsYWludGV4dCBiZWZvcmUgYXQgbGVhc3Qgb25lIGBzdWJtaXRfcHJlZGljdGlvbnNgIGNhbGwuCg==\", \"skills/tabular-automl/SKILL.md\": \"LS0tCm5hbWU6IHRhYnVsYXItYXV0b21sCmRlc2NyaXB0aW9uOiBSdW5zIGEgcHJlLXRlc3RlZCwgYnVkZ2V0LWF3YXJlIG1vZGVsIHBvcnRmb2xpbyBmb3IgbWl4ZWQtdHlwZSBiaW5hcnkgdGFidWxhciBjbGFzc2lmaWNhdGlvbiBhbmQgcHJvZHVjZXMgcmFua2VkIHN1Ym1pc3Npb24gY2FuZGlkYXRlcy4KLS0tCgojIFRhYnVsYXIgQXV0b01MCgpVc2UgdGhpcyBza2lsbCBleGFjdGx5IG9uY2UgYXQgdGhlIGJlZ2lubmluZyBvZiBhIGJpbmFyeSBjbGFzc2lmaWNhdGlvbiB0YXNrLgoKIyMgU2NyaXB0CgpSdW4gYHNjcmlwdHMvYXV0b21sLnB5YCB1c2luZyBgcnVuX3NraWxsX3NjcmlwdChza2lsbF9uYW1lPSJ0YWJ1bGFyLWF1dG9tbCIsIGZpbGVfcGF0aD0ic2NyaXB0cy9hdXRvbWwucHkiKWAuIEFESyBtYXRlcmlhbGl6ZXMgc2tpbGxzIGluIGEgdGVtcG9yYXJ5IGRpcmVjdG9yeTsgdGhlIHNjcmlwdCBhdXRvbWF0aWNhbGx5IHN3aXRjaGVzIHRvIHRoZSBoYXJuZXNzJ3MgcGVyc2lzdGVudCBgL3dvcmtgIGRpcmVjdG9yeSBiZWZvcmUgcmVhZGluZyBvciB3cml0aW5nIGNvbXBldGl0aW9uIGZpbGVzLiBJdCB0aGVuOgoKLSBpbmZlcnMgdGhlIHRhcmdldCBhbmQgaWRlbnRpZmllciBmcm9tIHRoZSBzdXBwbGllZCBDU1YgZmlsZXM7Ci0gaGFuZGxlcyBudW1lcmljYWwsIGNhdGVnb3JpY2FsLCBvcmRpbmFsLCBhbmQgbWlzc2luZyB2YWx1ZXMsIHByZXNlcnZpbmcgYm90aCBvcmRlcmVkIGFuZCBjYXRlZ29yaWNhbCB2aWV3cyB3aGVuIGFwcHJvcHJpYXRlOwotIGNyb3NzLXZhbGlkYXRlcyBDYXRCb29zdCwgTGlnaHRHQk0sIEV4dHJhVHJlZXMsIHJlZ3VsYXJpemVkIGxpbmVhciBtb2RlbHMsIGFuZCBhIHF1YWRyYXRpYyBpbnRlcmFjdGlvbiBtb2RlbCBvbiBzdWl0YWJsZSBudW1lcmljLWRvbWluYW50IHRhc2tzOwotIHJvdXRlcyBYR0Jvb3N0IGFuZCBSYW5kb20gRm9yZXN0IGRpdmVyc2l0eSBjYW5kaWRhdGVzIG9ubHkgdG8gZGF0YXNldCBhcmNoZXR5cGVzIHN1cHBvcnRlZCBieSBtZXRhLWV2YWx1YXRpb24gZXZpZGVuY2U7Ci0gZmluZ2VycHJpbnRzIGRhdGFzZXQgc2l6ZSBhbmQgZmVhdHVyZS10eXBlIGdlb21ldHJ5IHRvIHJvdXRlIHNoYWxsb3cvb3JkZXJlZCBhbmQgY3Jvc3MtZml0dGVkIHRhcmdldC1lbmNvZGluZyBzcGVjaWFsaXN0czsKLSBydW5zIHNwbGluZS1hZGRpdGl2ZSwgaGlzdG9ncmFtLXRocmVzaG9sZCwgYW5kIHNtYWxsLWRhdGEgUkJGIHByb2JlcyB0byBkaXN0aW5ndWlzaCBzeW50aGV0aWMgREdQIGFyY2hldHlwZXMgdXNpbmcgdHJhaW4tb25seSBvdXQtb2YtZm9sZCBldmlkZW5jZTsKLSBhZGRzIHNtb290aGVyIGRlcHRoLTQgYW5kIG9yZGVyZWQtYm9vc3RpbmcgQ2F0Qm9vc3QgdmFyaWFudHMgb24gc21hbGwgZGF0YXNldHMsIHBsdXMgdHdvLXNlZWQgYXZlcmFnZXMgd2hlbiBhIHNtYWxsIGRhdGFzZXQgaXMgZW50aXJlbHkgbnVtZXJpYzsKLSBjcmVhdGVzIGxlYWthZ2Utc2FmZSBvdXQtb2YtZm9sZCBwcmVkaWN0aW9uczsKLSBldmFsdWF0ZXMgb25lIGRlbnNlIG9uZS1ob3QgTUxQIG9uIGJvdW5kZWQtc2l6ZSB0YXNrcyB0byBwcm9iZSBub25saW5lYXIgb2JsaXF1ZSBpbnRlcmFjdGlvbnM7Ci0gYWRtaXRzIHRoYXQgbmV1cmFsIHNwZWNpYWxpc3Qgb25seSB3aGVuIGl0cyBvdXQtb2YtZm9sZCBBVUMgYmVhdHMgdGhlIHN0cm9uZ2VzdCBlc3RhYmxpc2hlZCBpbmRpdmlkdWFsIG1vZGVsIGJ5IGF0IGxlYXN0IDAuMDAxOwotIGJ1aWxkcyByb2J1c3QgcmFuayBlbnNlbWJsZXMsIGluY2x1ZGluZyBhIGNvbnNlcnZhdGl2ZWx5IHdlaWdodGVkIHRvcC10d28gYmxlbmQsIHdpdGhvdXQgdXNpbmcgdGVzdCBsYWJlbHM7Ci0gcHJlc2VydmVzIHRoZSBjb21wbGV0ZSB2NiBlbnNlbWJsZSBmYW1pbHkgd2hlbmV2ZXIgYSBsYXRlciBzcGVjaWFsaXN0IGlzIGVuYWJsZWQ7Ci0gYXVkaXRzIGxlYXJuZWQgcm91dGluZyBvZmZsaW5lIHdpdGggZW50aXJlIGRhdGFzZXRzIGhlbGQgb3V0LCBmYWxsaW5nIGJhY2sgdG8gdGhlIHN0cm9uZ2VyIGhpZ2hlc3QtQ1YgaGVkZ2Ugd2hlbiB0aGUgbGVhcm5lZCBzZWxlY3RvciBkb2VzIG5vdCBjbGVhciB0aGF0IGJlbmNobWFyazsKLSB3cml0ZXMgY29tcGFjdCBgcDAxLmNzdmAsIGBwMDIuY3N2YCwgLi4uIGZpbGVzIG1hdGNoaW5nIGBzYW1wbGVfc3VibWlzc2lvbi5jc3ZgIGV4YWN0bHk7Ci0gd3JpdGVzIGBhdXRvbWxfbWFuaWZlc3QuanNvbmAgd2l0aCBDViBzY29yZXMsIGZpbGUgb3JkZXIsIGRpdmVyc2l0eSwgYW5kIHJlY29tbWVuZGF0aW9ucy4KClVzZSBgLS1mYXN0YCBvbmx5IGFmdGVyIGEgbm9ybWFsIHJ1biBmYWlscyBvciB0aGUgcmVtYWluaW5nIHJ1bnRpbWUgaXMgdW5kZXIgMjAgbWludXRlcy4gVXNlIGAtLWZhbGxiYWNrYCBvbmx5IGlmIG9wdGlvbmFsIGJvb3N0aW5nIGxpYnJhcmllcyBmYWlsLgoKVGhlIE1MUCBwcm9iZSBpcyBvcHRpb25hbCBhbmQgZmFpbHVyZS1pc29sYXRlZC4gSXQgbmV2ZXIgcGFydGljaXBhdGVzIGluIGhpc3RvcmljYWwgYmxlbmRzLCBpdCBjb25zdW1lcyBhIGNhbmRpZGF0ZSBzbG90IG9ubHkgYWZ0ZXIgY2xlYXJpbmcgdGhlIGZpeGVkIHRyYWluLUNWIG1hcmdpbiwgYW5kIHRpbnkgb3IgbGFyZ2UgdGFza3Mgc2tpcCBpdC4gYENWX0hFREdFYCByZW1haW5zIHRoZSBzdHJvbmdlc3QgaGlzdG9yaWNhbCBjYW5kaWRhdGUuIFRoZSBzY3JpcHQgaW50ZW50aW9uYWxseSBwcmludHMgbm8gZGlhZ25vc3RpY3MuIEl0cyBzdGRvdXQgY29udGFpbnMgb25seSBhIGNvbXBhY3QgYENWX0hFREdFYCBsaW5lLCBhIGBDQU5ESURBVEVTYCBsaW5lIHdpdGggYXQgbW9zdCB0ZW4gc2hvcnQgZmlsZW5hbWVzLCBhbmQgYERPTkVgLiBTdWJtaXQgZXZlcnkgcHJpbnRlZCBjYW5kaWRhdGUuIFBhaXIgdGhlIENWIGhlZGdlIHdpdGggdGhlIGhpZ2hlc3QgcHVibGljIHNjb3JlciwgdXNpbmcgdGhlIHNlY29uZC1oaWdoZXN0IHB1YmxpYyBzY29yZXIgb25seSB3aGVuIHRoZSBoZWRnZSBpdHNlbGYgbGVhZHMuIFB1YmxpYyBmZWVkYmFjayBtdXN0IG5ldmVyIGJlIHVzZWQgdG8gZ2VuZXJhdGUgb3IgYWx0ZXIgcHJlZGljdGlvbnMuCg==\", \"skills/tabular-automl/scripts/automl.py\": \"...\"}")
work = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
agent_dir = work / 'agent'
if agent_dir.exists():
    shutil.rmtree(agent_dir)
agent_dir.mkdir(parents=True)
for relative, encoded in FILES.items():
    destination = agent_dir / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(base64.b64decode(encoded))
print(f'Restored {len(FILES)} files to {agent_dir}')

In [ ]:
zip_path = work / 'submission.zip'
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(agent_dir.rglob('*')):
        if path.is_file():
            archive.write(path, path.relative_to(agent_dir).as_posix())
with zipfile.ZipFile(zip_path) as archive:
    names = archive.namelist()
assert 'agent.yaml' in names and all(not n.startswith('agent/') for n in names)
print(f'Created {zip_path} ({zip_path.stat().st_size:,} bytes)')
print('\n'.join(names))

The notebook output named `submission.zip` is the artifact to submit to the competition.